In [1]:
import kagglehub

path = kagglehub.dataset_download("kazanova/sentiment140")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sentiment140' dataset.
Path to dataset files: /kaggle/input/sentiment140


In [2]:
import pandas as pd
import os

file_path = os.path.join(path, "training.1600000.processed.noemoticon.csv")

df = pd.read_csv(file_path, encoding='latin-1', header=None)
df.columns = ["sentiment", "id", "date", "flag", "user", "tweet"]
df.head()

import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from tqdm import tqdm
tqdm.pandas()

ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_tweet(tweet):
    tweet = re.sub(r'[^a-zA-Z]', ' ', tweet).lower().split()
    tweet = [ps.stem(word) for word in tweet if word not in stop_words]
    return ' '.join(tweet)

df['processed_tweet'] = df['tweet'].progress_apply(preprocess_tweet)


100%|██████████| 1600000/1600000 [02:15<00:00, 11769.89it/s]


In [6]:
df = df.sample(200_000, random_state=42).reset_index(drop=True)
df.shape

(200000, 7)

In [8]:
!pip install tensorflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 812.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=20000, oov_token="<UNK>")

tokenizer.fit_on_texts(df['processed_tweet'])
sequences = tokenizer.texts_to_sequences(df['processed_tweet'])
padded_sequences = pad_sequences(sequences, maxlen=100)

X = padded_sequences
df['sentiment'] = df['sentiment'].replace({4: 1})
y = df['sentiment'].astype(int)
vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary Size:", vocab_size)
df

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Vocabulary Size: 129088


,sentiment,id,date,flag,user,tweet,processed_tweet
0,1,2001831655,Tue Jun 02 02:13:29 PDT 2009,NO_QUERY,NinaMcFLY,@McFLYFan_Katie listening to wonderful music. ...,mcflyfan kati listen wonder music like holiday xx
1,1,2180626165,Mon Jun 15 10:43:32 PDT 2009,NO_QUERY,CrystalS7,is watching angels and demons &lt;3 for free ...,watch angel demon lt free http freetheat com
2,0,2301910371,Tue Jun 23 16:04:41 PDT 2009,NO_QUERY,pineviewca,spending time with my baby girl...before she g...,spend time babi girl goe colleg
3,1,2060616533,Sat Jun 06 19:14:36 PDT 2009,NO_QUERY,amongststars,I know you all too well,know well
4,1,2066463001,Sun Jun 07 10:34:04 PDT 2009,NO_QUERY,Little_Ren,@Burnt_feet hahaha cheers! I won't forget that...,burnt feet hahaha cheer forget exam love sopho...
...,...,...,...,...,...,...,...
199995,1,1972739635,Sat May 30 09:47:46 PDT 2009,NO_QUERY,AceMas21,@samanthai Awww *blush* Thanks!! You are a god...,samanthai awww blush thank goddess aswel
199996,0,2059413802,Sat Jun 06 16:53:06 PDT 2009,NO_QUERY,hannnnnnnah,"@therealTiffany awe, I'm sorry I hope you fee...",therealtiffani awe sorri hope feel better good...
199997,1,2176486732,Mon Jun 15 04:00:22 PDT 2009,NO_QUERY,LadanLashkari,"@AureliusTjin Haha, yeah. Your name is very u...",aureliustjin haha yeah name uniqu unless someo...
199998,0,2054128749,Sat Jun 06 06:37:48 PDT 2009,NO_QUERY,jessiiemcfly,i need money for tickets,need money ticket


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense
import tensorflow as tf

embd_dim = 128

with tf.device('/GPU:0'):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embd_dim, input_length=100),
        GRU(128, return_sequences=False),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(padded_sequences, y, epochs=5, batch_size=64)


Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


3125/3125 ━━━━━━━━━━━━━━━━━━━━ 652s 208ms/step - accuracy: 0.7338 - loss: 0.5236
Epoch 2/5
3028/3125 ━━━━━━━━━━━━━━━━━━━━ 17s 182ms/step - accuracy: 0.7984 - loss: 0.4303